# CNeuroMod QA — MRIQC metrics

Raincloud distributions of per-run image-quality metrics (framewise displacement, tSNR) across subjects and tasks, read from `output_data/qc_measures/`. Figures are written to `output_data/figures/qc_measures/`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
import seaborn as sns

try:
    import ptitprince
    HAS_PTITPRINCE = True
except Exception:
    HAS_PTITPRINCE = False

# Paths are provided by `invoke run-notebooks` as environment variables.
# Figures go in output_data/figures/{FIG_NAME}/ (also the notebook's "already
# ran" sentinel); the metric tables live in output_data/qc_measures|scans/.
FIG_NAME = "qc_measures"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data"))
FIG_DIR = OUTPUT_DIR / "figures" / FIG_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_tables(subdir):
    """Concatenate every non-empty TSV in output_data/<subdir>/."""
    frames = []
    for path in sorted((OUTPUT_DIR / subdir).glob("*.tsv")):
        try:
            frame = pd.read_csv(path, sep="\t")
        except (pd.errors.EmptyDataError, OSError):
            continue
        if not frame.empty:
            frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [ ]:
def raincloud(data, x, y, ax, palette="Set2"):
    """RainCloud (ptitprince) with a seaborn violin+strip fallback."""
    data = data[[x, y]].dropna()
    if data.empty:
        ax.set_title(f"no data for {y}")
        return ax
    if HAS_PTITPRINCE:
        try:
            ptitprince.RainCloud(x=x, y=y, hue=x, data=data, palette=palette,
                                 bw=.15, width_viol=.9, ax=ax, orient="v",
                                 alpha=.65, offset=-.05, move=.2, width_box=.1)
            if ax.get_legend():
                ax.get_legend().remove()
            return ax
        except Exception:
            ax.clear()
    sns.violinplot(data=data, x=x, y=y, ax=ax, hue=x, palette=palette,
                   inner=None, cut=0, legend=False)
    sns.stripplot(data=data, x=x, y=y, ax=ax, color="k", size=2, alpha=.4)
    return ax


In [ ]:
qc = load_tables("qc_measures")
print(f"loaded {len(qc)} runs")
if qc.empty:
    print("No QC metrics found — run `invoke run-qc-measures` first "
          "(needs data access).")
qc.head()

In [ ]:
if not qc.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    raincloud(qc, "subject", "fd_mean", ax)
    ax.set_xlabel("subject")
    ax.set_ylabel("Mean FD per run")
    fig.savefig(FIG_DIR / "fd_mean_by_subject.png", dpi=120, bbox_inches="tight")

In [ ]:
if not qc.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    raincloud(qc, "subject", "tsnr", ax)
    ax.set_xlabel("subject")
    ax.set_ylabel("Average tSNR per run")
    fig.savefig(FIG_DIR / "tsnr_by_subject.png", dpi=120, bbox_inches="tight")

In [ ]:
if not qc.empty and qc["task_grouped"].notna().any():
    fig, ax = plt.subplots(figsize=(14, 7))
    raincloud(qc, "task_grouped", "fd_mean", ax, palette="Set3")
    ax.tick_params("x", labelrotation=45)
    ax.set_xlabel("task")
    ax.set_ylabel("Mean FD per run")
    fig.savefig(FIG_DIR / "fd_mean_by_task.png", dpi=120, bbox_inches="tight")

In [ ]:
if not qc.empty and qc[["fd_mean", "tsnr"]].dropna().shape[0] > 0:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=qc, x="fd_mean", y="tsnr", hue="subject",
                    ax=ax, palette="Set2")
    ax.set_xlabel("Mean FD")
    ax.set_ylabel("Average tSNR")
    fig.savefig(FIG_DIR / "fd_vs_tsnr.png", dpi=120, bbox_inches="tight")